Note: 

Both SwapRateMCSimulator and SwapRatePCAEstimator require historical swap rate time series data in a specific format.
If no explicit file path is provided to the constructor, a default CSV file will be used instead.

The default dataset is located in the "curve_data" folder of the project (see "eur_swap_ts.csv" for the expected format).

In the examples below, a file path is explicitly provided, although it points to the same data as the default.

This notebook demonstrates how to:
- create an EURIBOR6M IRS payer/receiver position
- create a liability position, modelled as a portfolio of short position in Zero Coupon Bonds with maturity profile matching expected payment dates of    pension casfhlows (outflows)

In [11]:
from __future__ import annotations
import os 
from pathlib import Path

# It is assumed that the notebook is executed from within the project - either root or sub folder. 
CWD_PATH = Path.cwd()
SWAP_TIME_SERIES_CSV_PATH = ""

for parent in CWD_PATH.parents:
 if (parent / "src").exists():
     SWAP_TIME_SERIES_CSV_PATH = parent / "curve_data" / "eur_swap_ts.csv"
     break       
 else:
     SWAP_TIME_SERIES_CSV_PATH = CWD_PATH / "curve_data" / "eur_swap_ts.csv"

print(CWD_PATH)
print(SWAP_TIME_SERIES_CSV_PATH)   

C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\example_notebooks
C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\curve_data\eur_swap_ts.csv


In [12]:
# imports 
from typing import Dict
from datetime import datetime 
import numpy as np
import pandas as pd 
from rateslib import IRS as rlIRS, dcf as rlDCF 
from curve_engine.swap_curve_solver import SwapCurveSolver
from curve_engine.swap_curve_builder import SwapCurveBuilder
from position_engine.euribor6m_irs_position import Euribor6mIRSPosition
from position_engine.liability_position import LiabilityPosition 


In [13]:
# parameters 
curve_date = datetime(2026,3,31)

swap_rates =  {
 '2Y' : 2.948 , 
 '3Y' : 2.934, 
 '5Y' : 2.983, 
 '7Y' : 3.036, 
 '10Y': 3.11,  
 '12Y': 3.195,  
 '15Y': 3.2439, 
 '20Y': 3.246, 
 '25Y': 3.186,
 '30Y': 3.115 }

liabilities_payment_schedule = {
 '1Y': 240716648,   '2Y': 249313671,  '3Y': 260776369,  '4Y': 275104740,   
 '5Y': 292298787,   '6Y': 312358507,  '7Y': 335283902,  '8Y': 361074972, 
 '9Y': 384000367,  '10Y': 406925762, '11Y': 395463064, '12Y': 418388459,
 '13Y': 444179529, '14Y': 472836273, '15Y': 504358691, '16Y': 535881109, 
 '17Y': 561672178, '18Y': 578866225, '19Y': 587463248, '20Y': 590328922,
 '21Y': 584597573, '22Y': 590328922, '23Y': 584597573, '24Y': 573134876,
 '25Y': 555940830, '26Y': 533015435, '27Y': 504358691, '28Y': 469970598, 
 '29Y': 429851157, '30Y': 389731716 } 

In [14]:
# create a solver - enables calibration of curve to swap rates
solver = SwapCurveSolver(
 solver_id = 'eurswap_solver', 
 curve_date = curve_date, 
 instrument_rates = swap_rates
)

SUCCESS: `func_tol` reached after 7 iterations (levenberg_marquardt), `f_val`: 4.554749889445546e-16, `time`: 0.0962s


In [15]:
# construct the curve       
curve = SwapCurveBuilder.from_solver(
 curve_id = 'eurswap', 
 curve_solver = solver
)

In [16]:
# create the par -> forward Jacobian for the curve - needed to derive forward swap rates and calculate forward PV01
curve.calc_par_fwd_jacobian()

SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0081s


C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\.venv\Lib\site-packages\rateslib\curves\curves.py:275: UserWarning: Evaluating points on a curve beyond the endpoint of the basic spline interval is undefined.
date: 2056-04-07, spline end: 2056-04-05, curve id: 'eurswap_solver'
This often occurs when a curve is constructed with a final node date that aligns with the maturity of an instrument with a payment lag.
In the case that the instrument has a payment lag (e.g. a SOFR swap or ESTR swap or bond terminating on a non-business day) then a cashflow will occur after the maturity of the instrument.
The solution is to ensure that the final node date of the curve is changed to be beyond that expected payment date.
  warnings.warn(


In [17]:
# create irs 2y receiver position 
irs2y_receiver = Euribor6mIRSPosition(
  pos_id = 'EUR6M 2Y receiver', 
  pos_grp = 'hedge', 
  effective_dt = datetime(2026,3,31), 
  fixed_rate = 2.25, # (!) when is_par_swap = True, the fixed rate will be overridden and replaced by (mid-market) swap rate implied by the curve 
  notional = 1e6, # (!) notional sign does not impact receiver/payer type. IRS type is set explicitly via bool argument is_par_swap (see below)
  termination = '2Y', 
  is_receiver = True, 
  is_par_swap = True, 
  position_curve_builder = curve        
)

In [18]:
# create irs 2y payer position 
irs2y_payer = Euribor6mIRSPosition(
        pos_id = 'EUR6M 2Y payer', 
        pos_grp = 'hedge', 
        effective_dt = curve_date, 
        fixed_rate = 2.25, 
        notional = 1e6, 
        termination = '2Y', 
        is_receiver = False, 
        is_par_swap = True, 
        position_curve_builder = curve        
        )        

In [19]:
# create 30Y liability position 
lby_pos = LiabilityPosition(
  pos_id = 'liabilities 30Y', 
  pos_grp = 'liabilities', 
  currency = 'eur',   
  effective_dt = curve_date, 
  payment_schedule = liabilities_payment_schedule, 
  position_curve_builder = curve 
)

In [20]:
# irs attributes 
irs2y_receiver.notional
irs2y_receiver.fixed_rate
irs2y_receiver.maturity_dt  
irs2y_receiver.cashflows()
irs2y_receiver.spot_delta()
irs2y_receiver.fwd_delta()
irs2y_receiver.spot_gamma()
irs2y_receiver.npv()

irs2y_payer.notional
irs2y_payer.fixed_rate
irs2y_payer.maturity_dt  
irs2y_payer.cashflows()
irs2y_payer.spot_delta()
irs2y_payer.fwd_delta()
irs2y_payer.spot_gamma()
irs2y_payer.npv()

SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0042s
SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0039s


C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\.venv\Lib\site-packages\rateslib\curves\curves.py:275: UserWarning: Evaluating points on a curve beyond the endpoint of the basic spline interval is undefined.
date: 2056-04-07, spline end: 2056-04-05, curve id: 'eurswap_solver'
This often occurs when a curve is constructed with a final node date that aligns with the maturity of an instrument with a payment lag.
In the case that the instrument has a payment lag (e.g. a SOFR swap or ESTR swap or bond terminating on a non-business day) then a cashflow will occur after the maturity of the instrument.
The solution is to ensure that the final node date of the curve is changed to be beyond that expected payment date.
  warnings.warn(
C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\.venv\Lib\site-packages\rateslib\curves\curves.py:275: UserWarning: Evaluating points on a curve beyond the endpoint of the basic spline interval is undefined.
date: 2056-04-07, s

,Date,Position Grp,Position ID,Ccy,Curve,PV
0,2026-03-31,hedge,EUR6M 2Y payer,EUR,eurswap,0.0


In [21]:
# liability attributes 
lby_pos.currency
lby_pos.notional
lby_pos.fixed_rate 
lby_pos.maturity_dt
lby_pos.cashflows()  
lby_pos.spot_delta()
lby_pos.fwd_delta()
lby_pos.spot_gamma()
lby_pos.npv()
lby_pos.cashflow_zcb_positions

C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\.venv\Lib\site-packages\rateslib\curves\curves.py:275: UserWarning: Evaluating points on a curve beyond the endpoint of the basic spline interval is undefined.
date: 2056-04-07, spline end: 2056-04-05, curve id: 'eurswap_solver'
This often occurs when a curve is constructed with a final node date that aligns with the maturity of an instrument with a payment lag.
In the case that the instrument has a payment lag (e.g. a SOFR swap or ESTR swap or bond terminating on a non-business day) then a cashflow will occur after the maturity of the instrument.
The solution is to ensure that the final node date of the curve is changed to be beyond that expected payment date.
  warnings.warn(


SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0037s


,Cashflow Zero Coupon Bond,Notional,Ccy,Maturity Date,Maturity Years,Curve,DF,PV
0,ZCB-1,-240716648.0,EUR,2027-03-31,1.013889,eurswap,0.971280,-2.338033e+08
1,ZCB-2,-249313671.0,EUR,2028-03-31,2.030556,eurswap,0.943551,-2.352401e+08
2,ZCB-3,-260776369.0,EUR,2029-03-31,3.044444,eurswap,0.916845,-2.390916e+08
3,ZCB-4,-275104740.0,EUR,2030-03-31,4.058333,eurswap,0.890057,-2.448588e+08
4,ZCB-5,-292298787.0,EUR,2031-03-31,5.072222,eurswap,0.863229,-2.523207e+08
5,ZCB-6,-312358507.0,EUR,2032-03-31,6.088889,eurswap,0.836649,-2.613345e+08
6,ZCB-7,-335283902.0,EUR,2033-03-31,7.102778,eurswap,0.810783,-2.718424e+08
7,ZCB-8,-361074972.0,EUR,2034-03-31,8.116667,eurswap,0.785668,-2.836850e+08
8,ZCB-9,-384000367.0,EUR,2035-03-31,9.130556,eurswap,0.760725,-2.921189e+08
9,ZCB-10,-406925762.0,EUR,2036-03-31,10.147222,eurswap,0.735209,-2.991753e+08
